In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
!nvidia-smi

Sat Jan 31 16:51:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = "alxxtexxr/VRBI-Dataset"
allow_dir = 'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

...

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/49.8k [00:00<?, ?B/s]

Vision data downloaded to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2


In [4]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
import time

# Load all batch file paths
batch_paths = [f for f in vision_data_dir.glob('*.npz')]

# Initialize MiniBatchKMeans
k = 16384 # Number of clusters
kmeans = MiniBatchKMeans(
    n_clusters=k,
    init='random',
    batch_size=16384,
    n_init=1,
    reassignment_ratio=0.01,
    max_no_improvement=10,
    random_state=42
)

# Train in batches
# batch_paths -> current: 10 files
for i, batch_path in enumerate(batch_paths):
    print("="*128)
    print(f"Processing batch {i+1}/{len(batch_paths)}: {batch_path}")
    print("="*128)
    
    start_time = time.time()
    
    batch = np.load(batch_path)
    visual_embs = batch['visual_embs'].astype(np.float32) # (N, 2048) -> current: (32_400, 2048)
    
    # L2 normalize per vector
    visual_embs_unit = normalize(visual_embs, axis=1)
    
    print("Normalized visual embeddings shape:", visual_embs_unit.shape)
    print("Mean L2 norm (should be ~1):", np.linalg.norm(visual_embs_unit, axis=1).mean())
    print()
    
    # Update k-means
    kmeans.partial_fit(visual_embs_unit)
    
    end_time = time.time()
    iter_time = end_time - start_time
    
    print(f"Time taken: {iter_time:.2f}s")
    print()

Processing batch 1/10: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/400-499.vision_data.npz
Normalized visual embeddings shape: (32400, 2048)
Mean L2 norm (should be ~1): 0.9999999

Time taken: 53.30s

Processing batch 2/10: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/100-199.vision_data.npz
Normalized visual embeddings shape: (32400, 2048)
Mean L2 norm (should be ~1): 0.9999999

Time taken: 51.79s

Processing batch 3/10: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/600-699.vision_data.npz
Normalized visual embeddings shape: (32400, 2048)
Mean L2 norm (should be ~1): 0.9999999

Time taken: 51.81s

Processing batch 4/10: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/900-999.vision_data.npz
Normalized visual embeddings shape: (32400, 2048)
Mean L2 norm (should be ~1): 0.9999999

Time taken: 52.10s

Processing batch 5/10: /content/data

In [5]:
import joblib

centroids_unit = kmeans.cluster_centers_

kmeans_path = vision_data_dir / 'kmeans.pkl'
centroids_unit_path = vision_data_dir / 'centroids_unit.npy'

joblib.dump(kmeans, kmeans_path)
np.save(centroids_unit_path, centroids_unit)

print("K-means model saved to:", kmeans_path)
print("Normalized centroids saved to:", centroids_unit_path)

K-means model saved to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/kmeans.pkl
Normalized centroids saved to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/centroids_unit.npy


In [6]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj=str(kmeans_path),
    path_in_repo=str(kmeans_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)
upload_file(
    path_or_fileobj=str(centroids_unit_path),
    path_in_repo=str(centroids_unit_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tions_train.v2/kmeans.pkl:   0%|          |  552kB /  134MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ain.v2/centroids_unit.npy:  25%|##4       | 33.4MB /  134MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset/commit/e5f0a50c6736e5ad9a616722d0975bb9301cee4f', commit_message='Upload model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/centroids_unit.npy with huggingface_hub', commit_description='', oid='e5f0a50c6736e5ad9a616722d0975bb9301cee4f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/VRBI-Dataset'), pr_revision=None, pr_num=None)